# Contextual gene embeddings -> pathway activity (decoupler hallmark ULM)

Builds a cells x genes matrix from scOntoFM contextual gene embeddings, scores MSigDB hallmark
pathways per cell with decoupler (ULM), and plots pathway activity per cell type.

Prerequisite (from the repository root), e.g. for the Kim et al. 2020 lung dataset:

```
python scripts/run_inference.py --input_h5ad data/Kim2020_Lung_convert.h5ad \
    --run_id scOntoFM --save_gene_embeddings --save_data
```

This writes `output/scOntoFM/<dataset_id>/` with `<dataset_id>.h5ad` (with UMAP), `<dataset_id>_gene_embeddings.npy`
and `<dataset_id>_hf_dataset/`. `dc.op.hallmark` downloads the hallmark gene sets (internet access required).
Figures are written to `./vis_result/`.

In [ ]:
import os
import sys
import pickle
from pathlib import Path

import scanpy as sc
import decoupler as dc
import numpy as np
from datasets import Dataset
import matplotlib.pyplot as plt
import matplotlib as mpl

REPO_ROOT = Path.cwd().resolve().parents[1]  # notebook lives in downstream/3_gene_function
sys.path.insert(0, str(REPO_ROOT))
from scOntoFM.plotting import config_rc, centimeter
from scOntoFM.paths import TOKEN_DICT_PATH

# inference outputs: <OUTPUT_ROOT>/<RUN_ID>/<dataset_id>/
OUTPUT_ROOT = REPO_ROOT / "output"
RUN_ID = "scOntoFM"
os.makedirs("vis_result", exist_ok=True)

config_rc(dpi=300, font_size=5, lw=0.5)
sc.set_figure_params(facecolor="white", figsize=(centimeter*6, centimeter*6), dpi_save=300, fontsize=5, vector_friendly=True)

In [ ]:
#dataset_id = "pbmc10k_convert"
dataset_id = "Kim2020_Lung_convert"

path = OUTPUT_ROOT / RUN_ID / dataset_id / f"{dataset_id}.h5ad"

adata = sc.read_h5ad(path)
adata

In [ ]:
fig, ax = plt.subplots(
    figsize=(centimeter * 4, centimeter * 4),
    dpi=300
)

sc.pl.umap(
    adata,
    color="cell_type", #str_labels
    edges_color='none',
    title=f"Cell Embeddings colored by celltype",
    frameon=False,
    ax = ax
)

# save to file
fig.savefig(
    "vis_result/lung_cluster.pdf",    # or "violin_plot.pdf", "violin_plot.svg"
    dpi=300,              # match your desired resolution
    bbox_inches="tight",  # trims extra whitespace
)
plt.show()

In [ ]:
# remove Other celltype for pbmc10k
adata = adata[adata.obs['cell_type'] != "Other", :].copy()
adata

In [ ]:
path = OUTPUT_ROOT / RUN_ID / dataset_id / f"{dataset_id}_gene_embeddings.npy"

gene_emb_matrix = np.load(path)
gene_emb_matrix.shape

In [ ]:
# load the tokenized (HF) dataset
base_path = OUTPUT_ROOT / RUN_ID / dataset_id
print(f"Loading dataset from {base_path} ...")
dataset = Dataset.load_from_disk(os.path.join(base_path, f"{dataset_id}_hf_dataset"))

In [ ]:
# load token dict
token_dict_path = TOKEN_DICT_PATH
with open(token_dict_path, "rb") as f:
    token_dict = pickle.load(f)

In [ ]:
token_to_gene = {v: k for k, v in token_dict.items()}
gene_to_varidx = {gene: i for i, gene in enumerate(adata.var_names)}

token_to_varidx = {
    tok: gene_to_varidx[token_to_gene[tok]]
    for tok in token_to_gene
    if token_to_gene[tok] in gene_to_varidx
}

In [ ]:
import numpy as np
from scipy.sparse import coo_matrix

num_cells = len(dataset)
num_genes = adata.n_vars

rows = []
cols = []
data = []

for cell_idx, example in enumerate(dataset):
    # remove CLS/SEP etc
    input_ids = np.array(example["input_ids"][1:-1])
    seq_len = len(input_ids)

    emb = gene_emb_matrix[cell_idx, :seq_len]  # (seq_len,)

    valid_mask = np.array([tok in token_to_varidx for tok in input_ids])
    if not valid_mask.any():
        continue

    valid_ids = input_ids[valid_mask]               # (n_valid,)
    valid_emb = emb[valid_mask]                     # (n_valid,)
    col_indices = [token_to_varidx[t] for t in valid_ids]

    rows.extend([cell_idx] * len(col_indices))
    cols.extend(col_indices)
    data.extend(valid_emb.tolist())                 # scalar per gene

from scipy.sparse import csr_matrix

mat = csr_matrix(
    (data, (rows, cols)),
    shape=(num_cells, num_genes)
)

In [ ]:
adata.X = mat

In [ ]:
adata

In [ ]:
adata.X

In [ ]:
import numpy as np
import umap
from sklearn.decomposition import PCA

X = adata.X  # (cells × genes)
X = X.toarray() if not isinstance(X, np.ndarray) else X

X_T = X.T    # (genes × cells)
pca = PCA(n_components=50)
gene_pca = pca.fit_transform(X_T)   # (genes × 50)
reducer = umap.UMAP(n_neighbors=10, min_dist=0.3)
gene_umap = reducer.fit_transform(gene_pca)  # (genes × 2)

import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.scatter(gene_umap[:,0], gene_umap[:,1], s=4)
plt.title("Gene-level UMAP")
plt.show()

In [ ]:
adata.var_names = adata.var['gene_symbols'] #gene_symbols

In [ ]:
# for lung data, show hallmark gene
hallmark = dc.op.hallmark(organism="human")
hallmark

In [ ]:
dc.mt.ulm(data=adata, net=hallmark)


score = dc.pp.get_obsm(adata=adata, key="score_ulm")
score

In [ ]:
df = dc.tl.rankby_group(adata=score, groupby="cell_type", reference="rest", method="t-test_overestim_var")
# str_labels
df

In [ ]:
top5 = (
    df.sort_values(["group", "stat"], ascending=[True, False])
      .groupby("group")
      .head(5)
      .reset_index(drop=True)
)

top5

In [ ]:
score.X

In [ ]:
fig, ax = plt.subplots(
    figsize=(centimeter * 4, centimeter * 4),
    dpi=300
)
#pathway_name = "ALLOGRAFT_REJECTION" # vmax=15
#pathway_name = "COAGULATION" # vmax=10
pathway_name = "GLYCOLYSIS"  #vmax=5,
sc.pl.umap(score, color=[pathway_name], cmap="RdBu_r", vmax=9, ax=ax)
# After sc.pl.violin(...)
for spine in ax.spines.values():
    spine.set_linewidth(0.5)  # pick your thickness, e.g., 0.8 or 1.2

# (optional) adjust tick mark linewidth too
ax.tick_params(axis="both", width=0.5, length=2)

# Get the colorbar from the first PathCollection
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(length=2, width=0.5, direction="out")  # tweak as desired

fig.savefig(
    f"vis_result/cluster_heatmap_{pathway_name}.pdf",    # or "violin_plot.pdf", "violin_plot.svg"
    dpi=300,              # match your desired resolution
    bbox_inches="tight",  # trims extra whitespace
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

#pathway_name = "ALLOGRAFT_REJECTION" # vmax=15
#pathway_name = "COAGULATION" # vmax=10
pathway_name = "GLYCOLYSIS"  #vmax=5,


fig, ax = plt.subplots(
    figsize=(centimeter * 6, centimeter * 4),
    dpi=300
)
sc.pl.violin(
    score,
    keys=pathway_name,
    groupby="cell_type",
    rotation=45,
    stripplot=True,   # add the points
    jitter=0.1,         # 0 for no jitter; set a small value if you want a tiny spread
    size=0.2,           # point size
    color= "gray",     # violin color
    ax=ax,
    show=False,       # keep control of plt.show
)

# After sc.pl.violin(...)
for spine in ax.spines.values():
    spine.set_linewidth(0.5)  # pick your thickness, e.g., 0.8 or 1.2

# (optional) adjust tick mark linewidth too
ax.tick_params(axis="both", width=0.5, length=2)


# Recolor only the strip points (PathCollection), make them gray & semi-transparent
for coll in ax.collections:
    if isinstance(coll, mpl.collections.PathCollection):
        coll.set_facecolor((0.5, 0.5, 0.5, 0.4))  # RGBA
        coll.set_edgecolor((0.5, 0.5, 0.5, 0.4))
        
# After sc.pl.violin(...)
for label in ax.get_xticklabels():
    label.set_horizontalalignment("right")  # center on the tick
    label.set_rotation(45)                   # keep your rotation
    label.set_rotation_mode("anchor")        # avoids shifting when rotated
    label.set_verticalalignment("center")    # optional, for vertical centering

# save to file
fig.savefig(
    f"vis_result/violin_plot_{pathway_name}.pdf",    # or "violin_plot.pdf", "violin_plot.svg"
    dpi=300,              # match your desired resolution
    bbox_inches="tight",  # trims extra whitespace
)
plt.show()

In [ ]:
score.var_names